In [ ]:
import numpy as np

rng = np.random.default_rng(7)    
DIM = 128                          
RUIDO = 0.05                       
UMBRAL = 1.0                       

def normalizar(v):
    """Lleva el vector a norma 1 (como hace FaceNet)."""
    return v / np.linalg.norm(v)

prototipos = {nombre: normalizar(rng.normal(size=DIM))
              for nombre in ["Ana", "Luis", "María"]}

def foto_de(nombre):
    """Simula el embedding de una foto: prototipo de la persona + ruido."""
    return normalizar(prototipos[nombre] + rng.normal(scale=RUIDO, size=DIM))

def distancia(a, b):
    """Distancia euclídea entre dos vectores."""
    return float(np.linalg.norm(a - b))

print("Preparado. Personas registradas:", list(prototipos.keys()))

Preparado. Personas registradas: ['Ana', 'Luis', 'María']


In [2]:
# Arquitectura de la red siamesa (solo de referencia).
# Si no tienes TensorFlow, este bloque avisa y el ejemplo continúa igual.
try:
    import tensorflow as tf
    from tensorflow.keras import layers, Model

    def red_base():
        e = layers.Input(shape=(64, 64, 3))
        x = layers.Conv2D(32, 3, activation="relu", padding="same")(e)
        x = layers.MaxPooling2D()(x)
        x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
        x = layers.GlobalAveragePooling2D()(x)
        x = layers.Dense(128)(x)
        x = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1))(x)
        return Model(e, x, name="red_base")

    base = red_base()
    a = layers.Input(shape=(64, 64, 3)); b = layers.Input(shape=(64, 64, 3))
    # La MISMA red base procesa las dos entradas -> pesos compartidos
    dist = layers.Lambda(lambda v: tf.norm(v[0] - v[1], axis=1))([base(a), base(b)])
    red_siamesa = Model([a, b], dist, name="red_siamesa")
    red_siamesa.summary()
except Exception as e:
    print("(TensorFlow no disponible; se omite la arquitectura y se sigue con el ejemplo.)")

Model: "red_siamesa"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_2       │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ red_base            │ (None, 128)       │     27,712 │ input_layer_1[0]… │
│ (Functional)        │                   │            │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_1 (Lambda)   │ (None)            │          0 │ red_base[0][0],   │
│                     │                   │            │ red_base[1][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 27,712 (108.25 KB)

 Trainable params: 27,712 (108.25 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
# Verificación (one-shot): comparar dos fotos y decidir con el umbral.

def verificar(foto_nueva, foto_registrada, umbral=UMBRAL):
    d = distancia(foto_nueva, foto_registrada)
    return d < umbral, d

# Caso 1: dos fotos de la MISMA persona
igual, d1 = verificar(foto_de("Ana"), foto_de("Ana"))
print(f"Ana vs Ana   -> distancia {d1:.3f} | ¿misma persona? {igual}")

# Caso 2: fotos de personas DISTINTAS
igual, d2 = verificar(foto_de("Ana"), foto_de("Luis"))
print(f"Ana vs Luis  -> distancia {d2:.3f} | ¿misma persona? {igual}")

Ana vs Ana   -> distancia 0.699 | ¿misma persona? True
Ana vs Luis  -> distancia 1.357 | ¿misma persona? False


In [4]:
# --- DeepFace: pérdida contrastiva (trabaja con PARES) ---
def perdida_contrastiva(dist, misma_persona, margen=1.0):
    y = 1.0 if misma_persona else 0.0
    iguales = y * dist**2
    distintos = (1 - y) * max(margen - dist, 0)**2
    return iguales + distintos

par_igual = distancia(foto_de("Ana"), foto_de("Ana"))     # deberían estar cerca
par_distinto = distancia(foto_de("Ana"), foto_de("Luis"))  # deberían estar lejos

print("DeepFace (pares):")
print(f"  par igual    -> distancia {par_igual:.3f} | pérdida {perdida_contrastiva(par_igual, True):.3f}")
print(f"  par distinto -> distancia {par_distinto:.3f} | pérdida {perdida_contrastiva(par_distinto, False):.3f}")

DeepFace (pares):
  par igual    -> distancia 0.695 | pérdida 0.482
  par distinto -> distancia 1.443 | pérdida 0.000


In [5]:
# --- FaceNet: triplet loss (trabaja con TRIPLETAS) ---
def triplet_loss(ancla, positivo, negativo, margen=0.2):
    d_pos = distancia(ancla, positivo)   # ancla vs misma persona
    d_neg = distancia(ancla, negativo)   # ancla vs otra persona
    return max(d_pos**2 - d_neg**2 + margen, 0), d_pos, d_neg

ancla = foto_de("Ana")
positivo = foto_de("Ana")     # misma persona que el ancla
negativo = foto_de("Luis")    # persona distinta

perdida, d_pos, d_neg = triplet_loss(ancla, positivo, negativo)
print("FaceNet (tripletas):")
print(f"  ancla-positivo -> {d_pos:.3f}   (se quiere pequeña)")
print(f"  ancla-negativo -> {d_neg:.3f}   (se quiere grande)")
print(f"  triplet loss   -> {perdida:.3f}  (0 significa que ya cumple el objetivo)")

FaceNet (tripletas):
  ancla-positivo -> 0.681   (se quiere pequeña)
  ancla-negativo -> 1.396   (se quiere grande)
  triplet loss   -> 0.000  (0 significa que ya cumple el objetivo)


In [6]:
# Verificación (1:1): UNA comparación
def verificacion_1a1(foto_nueva, nombre_declarado, umbral=UMBRAL):
    d = distancia(foto_nueva, prototipos[nombre_declarado])
    return ("aceptado" if d < umbral else "rechazado"), d, 1  # 1 = nº de comparaciones

resultado, d, comparaciones = verificacion_1a1(foto_de("Ana"), "Ana")
print(f"Verificación (dice ser Ana): {resultado} | distancia {d:.3f} | comparaciones: {comparaciones}")

Verificación (dice ser Ana): aceptado | distancia 0.513 | comparaciones: 1


In [7]:
# Reconocimiento (1:N): N comparaciones y se elige el vecino más cercano
def reconocimiento_1aN(foto_nueva, umbral=UMBRAL):
    mejor_nombre, mejor_dist, comparaciones = None, float("inf"), 0
    for nombre, vector in prototipos.items():
        d = distancia(foto_nueva, vector)
        comparaciones += 1                       # cada vuelta es una comparación
        if d < mejor_dist:
            mejor_dist, mejor_nombre = d, nombre
    identidad = mejor_nombre if mejor_dist < umbral else "desconocido"
    return identidad, mejor_dist, comparaciones

# Alguien que sí está en la base
ident, d, comps = reconocimiento_1aN(foto_de("María"))
print(f"Reconocimiento -> {ident} | distancia {d:.3f} | comparaciones: {comps}")

# Alguien que NO está en la base (vector al azar)
ident, d, comps = reconocimiento_1aN(normalizar(rng.normal(size=DIM)))
print(f"Reconocimiento -> {ident} | distancia {d:.3f} | comparaciones: {comps}")

Reconocimiento -> María | distancia 0.502 | comparaciones: 3
Reconocimiento -> desconocido | distancia 1.273 | comparaciones: 3


In [8]:
# El costo: la verificación siempre hace 1 comparación; el reconocimiento hace N.
print("Comparaciones necesarias según el tamaño de la base (N):")
for N in [1, 100, 10_000, 1_000_000]:
    print(f"  N = {N:>9}  ->  verificación: 1   |   reconocimiento: {N}")

Comparaciones necesarias según el tamaño de la base (N):
  N =         1  ->  verificación: 1   |   reconocimiento: 1
  N =       100  ->  verificación: 1   |   reconocimiento: 100
  N =     10000  ->  verificación: 1   |   reconocimiento: 10000
  N =   1000000  ->  verificación: 1   |   reconocimiento: 1000000
